# 10 — Live Demo: 2025-26 NBA Playoffs

Diese Saison laeuft noch — Round 1 ist gerade im Gange. Die Frage:

> Welcher Team gibt unser Modell *jetzt* die beste Title-Chance?

Wir nutzen die Module aus `src/` (statt alles im Notebook zu wiederholen). Das demonstriert die saubere Code-Struktur.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))
from src import bracket, series, plot_style
plot_style.apply()

DATA = Path('..') / 'data' / 'processed'
df = pd.read_parquet(DATA / 'games_with_advanced_features.parquet')

SEASON = 2025
season_games = df[df.season == SEASON]
playoffs = season_games[season_games.gameType == 'Playoffs']
print(f'Saison {SEASON}: {len(season_games):,} Spiele, davon {len(playoffs)} Playoff-Spiele')
print(f'Letztes Spiel im Datensatz: {season_games.gameDate.max().date()}')

## Aktuelle ELO-Rangliste (Top 20)

Pre-Playoff ELO-Stand jedes Teams — die wichtigste Eingangsgroesse fuer alle Simulationen.

In [ ]:
names = pd.concat([
    df[['gameDate', 'hometeamId', 'hometeamName']].rename(columns={'hometeamId': 't', 'hometeamName': 'n'}),
    df[['gameDate', 'awayteamId', 'awayteamName']].rename(columns={'awayteamId': 't', 'awayteamName': 'n'}),
]).sort_values('gameDate').drop_duplicates('t', keep='last').set_index('t')['n']

home_v = playoffs[['gameDate', 'hometeamId', 'home_elo_pre']].rename(columns={'hometeamId': 't', 'home_elo_pre': 'e'})
away_v = playoffs[['gameDate', 'awayteamId', 'away_elo_pre']].rename(columns={'awayteamId': 't', 'away_elo_pre': 'e'})
preplay = pd.concat([home_v, away_v]).sort_values(['t', 'gameDate']).groupby('t').first()['e']

elo_table = preplay.rename(index=names).sort_values(ascending=False).head(20)
elo_table.round(0).to_frame('Pre-Playoff ELO')

## Aktuelle R1-Paarungen + Live-Stand

In [ ]:
playoffs = playoffs.copy()
playoffs['team_pair'] = playoffs.apply(lambda r: tuple(sorted([r.hometeamId, r.awayteamId])), axis=1)

round1 = []
for pair, grp in playoffs.groupby('team_pair'):
    higher = grp.hometeamId.value_counts().idxmax()
    lower = [t for t in pair if t != higher][0]
    wins_h = ((grp.hometeamId == higher) & (grp.home_win == 1)).sum() + \
             ((grp.awayteamId == higher) & (grp.home_win == 0)).sum()
    wins_l = len(grp) - wins_h
    round1.append({
        'higher': names.get(higher, str(higher)),
        'lower':  names.get(lower, str(lower)),
        'higher_elo': round(preplay.get(higher, 1500)),
        'lower_elo':  round(preplay.get(lower, 1500)),
        'series_score': f'{max(wins_h, wins_l)}-{min(wins_h, wins_l)}',
        'leader': names.get(higher, '?') if wins_h > wins_l else (names.get(lower, '?') if wins_l > wins_h else 'gleich'),
        'games_played': len(grp),
    })
pd.DataFrame(round1).sort_values('higher_elo', ascending=False)

## Hypothetische Championship-Probabilities (ELO-basiert)

Wenn der Bracket noch nicht steht, nehmen wir die **Top-16-Teams nach ELO** und sortieren sie nach klassischem 1v8 / 4v5 / 2v7 / 3v6 Matchup-Schema. Eine plausible Approximation, was das Modell tippen wuerde.

In [ ]:
top16 = preplay.sort_values(ascending=False).head(16)
elos = top16.to_dict()
team_ids = list(top16.index)

seed_matchups = [(0, 7), (3, 4), (1, 6), (2, 5), (8, 15), (11, 12), (9, 14), (10, 13)]

rng = np.random.default_rng(42)
champ_counts = {t: 0 for t in team_ids}
N_SIM = 10000
for _ in range(N_SIM):
    r1_winners = []
    for hi, lo in seed_matchups:
        ta, tb = team_ids[hi], team_ids[lo]
        r1_winners.append(ta if series.simulate_b07_elo(elos[ta], elos[tb], rng=rng) else tb)
    r2_winners = []
    for i in range(0, 8, 2):
        ta, tb = r1_winners[i], r1_winners[i+1]
        higher, lower = (ta, tb) if elos[ta] >= elos[tb] else (tb, ta)
        r2_winners.append(higher if series.simulate_b07_elo(elos[higher], elos[lower], rng=rng) else lower)
    r3_winners = []
    for i in range(0, 4, 2):
        ta, tb = r2_winners[i], r2_winners[i+1]
        higher, lower = (ta, tb) if elos[ta] >= elos[tb] else (tb, ta)
        r3_winners.append(higher if series.simulate_b07_elo(elos[higher], elos[lower], rng=rng) else lower)
    ta, tb = r3_winners
    higher, lower = (ta, tb) if elos[ta] >= elos[tb] else (tb, ta)
    champ = higher if series.simulate_b07_elo(elos[higher], elos[lower], rng=rng) else lower
    champ_counts[champ] += 1

probs = pd.Series({names.get(t, str(t)): c / N_SIM for t, c in champ_counts.items()}).sort_values(ascending=False)
print('Top-5 Championship-Picks fuer 2025-26:')
print(probs.head(5).apply(lambda v: f'{v:.1%}'))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
view = probs.sort_values()
ax.barh(view.index, view.values, color=plot_style.COLORS['primary'])
ax.set_xlabel('P(Championship)')
ax.set_title(f'Live: 2025-26 NBA Championship Probabilities\n(Stand: {season_games.gameDate.max().date()}, Top-16 nach ELO)')
for i, v in enumerate(view.values):
    ax.text(v + 0.005, i, f'{v:.1%}', va='center', fontsize=9)
ax.set_xlim(0, max(view.values) * 1.18)
plt.tight_layout()
plt.show()

## Was diese Demo zeigt

- Live-Vorhersagen sind mit dem Modell **trivial reproduzierbar** — wir haben den ganzen Pipeline in `src/` extrahiert und brauchen hier nur ~30 Zeilen Notebook-Code.
- Die ELO-basierte Bracket-Sim laeuft in Sekunden.

Caveat: Das ist ein einfaches ELO-only Live-Setup. Fuer eine richtige Production-Version wuerde man:
- alle Advanced Features (Player Box-Score, SoS, Star Availability) als Snapshot nachladen
- die echten Round-1-Paarungen aus aktuellen Daten verwenden
- nach jedem realen Spiel re-runnen, um Conditional Predictions zu aktualisieren